# Autonomous Incident Response System for AWS
---

In this example, we will build an Agentic system to respond to incidents in your AWS accounts. This is a multi-agent system that composes 4 main components: 

1. **Monitoring**: This composes of a couple of aspects which includes monitoring CloudWatch alarms on the pre-built alarms that have already been set in your account. This might include high `CPU` usage, unhealthy load balancers, SageMaker instance cost allocations, etc. This would also include observing logs from different services from your account and classifying those logs into `Critical` (for example service down, `CPU`>`90%`), `Warning` (for example, latency > threshold, or if something goes beyond a threshold for a specific service) and `Informational` (for example, routine backups, information on various running applications in the AWS account, etc.).

1. **Diagnosis**: This includes diagnosis events that are seen through the monitoring agent. This can include querying `AWS` CloudTrail for additional data, X-Ray data and document these findings in reports that can be saved and used later in the resolution process. This would contain information only on the errors and the different services that need a resolution.

1. **Resolution**: This portion of the solution will be triggered by a diagnosis done from the step before. Once the diagnoses is done with the clear report, then this portion starts to remediate certain actions, such as adjusting EC2 auto-scaling group capacities, invoking functions to rollback deployments, etc. This agent is an essential part of the system since it will be using AWS `API`s in real time to manage the resources.

1. **Communication**: Last, this agent is responsible for keeping track of updates, creating and updating tickets in Jira, sending real time notifications to Slack with the incident details and resolution updates.

This solution will also contain aspects for observabilitiy and tracing but without further ado, let's get right into it.

In [1]:
# LangGraph is a low level orchestration framework for building controllable agents. 
# While langchain provides integrations and composable components to streamline LLM application development, 
# the LangGraph library enables agent orchestration, long term memory, human in the loop and customizable architectures.

In [5]:
import logging
from typing import Annotated, List, Dict, Any
from typing_extensions import TypedDict
# import langgraph relevant libraries
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from datetime import datetime

# Import the memory saver to save in checkpoint and in some thread to retain agent's memory
from langgraph.checkpoint.memory import MemorySaver

# langchain imports
from langchain_aws.chat_models import ChatBedrockConverse
from langchain_core.tools import tool

In [6]:
# Create a logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers
logger.handlers.clear()

# Add a simple handler
handler = logging.StreamHandler()
formatter = logging.Formatter('[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

In [9]:
# define the constants
AMAZON_NOVA_PRO_MODEL_ID: str = 'amazon.nova-pro-v1:0'

In [10]:
# create the state schema which will track the AWS incident responses
import json
import time
from typing import List

class AWSState(TypedDict):
    """
    This class contains the state for the AWS incident response.
    """
    # the messages adds it to the list of messages that preserves
    # the chat history
    messages: Annotated[list, add_messages]
    # This incident contains the incident details
    incident: dict
    # Contains the results from the diagnosis
    diagnosis: dict
    # Contains the list of resolution actions that needs to be performed
    actions: list
    # to store communication records and the status of the incident
    notifications: list
    status: str

In [12]:
# Let's now initialize our LLM, we will use Amazon Nova for this example
llm = ChatBedrockConverse(
    model_id=AMAZON_NOVA_PRO_MODEL_ID,
    temperature = 0.1,
)

[2025-03-14 12:39:04,037] p18179 {credentials.py:1352} INFO - Found credentials in shared credentials file: ~/.aws/credentials
